# 04 · Discretisation

Turns cleaned numbers into items such as `CHL_COP_1=[4,6)` so Apriori can count them (paper Section 3.3, step 2).

1. **Design breakpoints** from engineering limits (see `DESIGN_BREAKPOINTS` in `src/discretise.py`).
2. **Entropy refinement** (paper Eq. 7, η = 1.5 %): inside each design interval, keep a split only if it reduces the entropy of the **efficiency band** (plant COP low / medium / high) by at least 1.5 %.
3. Variables with neither get **tertiles** (low / medium / high); on/off columns become `on` / `off`.

Breakpoints are learned on the **fault-free** file only and applied unchanged to every fault file.

In [ ]:
import os, sys, json
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
from src import discretise as dz

S = dz.load_settings("config.yaml")
pd.set_option("display.width", 200, "display.max_colwidth", 120)
S

## Load the cleaned fault-free data
Uses Member A's output in `data/processed/`. If it is still 1-minute data, it is averaged to 15 minutes here and only rows with at least one chiller on are kept.

In [ ]:
def load_processed(stem):
    for ext, reader in [(".parquet", pd.read_parquet), (".pkl", pd.read_pickle)]:
        p = Path("data/processed") / f"{stem}{ext}"
        if p.exists():
            return reader(p)
    raise FileNotFoundError(f"data/processed/{stem}.parquet or .pkl not found - run the preprocessing notebook first")

def to_15min(df):
    step = df.index.to_series().diff().median()
    if step < pd.Timedelta("15min"):
        df = df.resample("15min").mean()
        sta = [c for c in df.columns if c.endswith(tuple(f"_STA_{i}" for i in range(1, 4)))]
        df[sta] = (df[sta] > 0.5).astype(float)          # status: on if on most of the 15 min
    chl = [c for c in df.columns if c.startswith("CHL_STA_")]
    return df[df[chl].max(axis=1) > 0.5]                 # keep rows with a chiller on

ff = to_15min(load_processed("ChillerPlant"))
print(ff.shape)

## Fit the breakpoints (fault-free year only)

In [ ]:
specs = dz.fit_breakpoints(ff, s=S)
summary = dz.describe(specs, ff)
print(len(specs), "variables discretised")
summary["method"].value_counts()

## Five example variables
- `CHL_SW_TEMP_1`: design limits 42 / 52 °F (set-point reset range)
- `CT_APPROACH_1`: design approach 8 °F
- `CHL_COP_1`: paper's normal COP range 4–6
- `OA_TEMP_WB`: no design limit → entropy splits or tertiles
- `CWL_SEC_PM_SPD_1`: pump speed (0–1), no design limit

In [ ]:
examples = ["CHL_SW_TEMP_1", "CT_APPROACH_1", "CHL_COP_1", "OA_TEMP_WB", "CWL_SEC_PM_SPD_1"]
summary[summary["variable"].isin(examples)].set_index("variable").loc[[e for e in examples if e in specs]]

## All variables

In [ ]:
summary

## Save the breakpoints (used for every file)

In [ ]:
out = Path("data/processed/breakpoints.json")
out.write_text(json.dumps(specs, indent=2))
print("saved", out)

## Transactions

In [ ]:
tx = dz.to_transactions(dz.discretise(ff, specs))
print("transactions:", len(tx))
print("items per transaction (mean):", round(np.mean([len(t) for t in tx]), 1))
print("distinct items:", len(set().union(*tx)))
sorted(tx[len(tx) // 2])[:15]

## Same breakpoints on a fault file (consistency check)

In [ ]:
fault = to_15min(load_processed("ChillerPlant_coolingtower_fouling_065"))
d_ff = dz.discretise(ff, specs)["CT_APPROACH_1"].value_counts(normalize=True).rename("fault-free")
d_ft = dz.discretise(fault, specs)["CT_APPROACH_1"].value_counts(normalize=True).rename("tower fouling 65%")
pd.concat([d_ff, d_ft], axis=1).round(3)   # fouling should push CT_APPROACH_1 into the higher intervals